In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.features.plate_discipline import add_discipline_flags
from src.features.batted_ball import batted_ball_events, add_quality_flags

df = load_all_snapshots(seasons=[2024])
f = add_discipline_flags(df)
bbe = add_quality_flags(batted_ball_events(df))
bbe = bbe[bbe["pitch_type"].notna()]

damage = pd.DataFrame({
    "bbe": bbe.groupby(["batter", "pitch_type"]).size(),
    "barrels": bbe.groupby(["batter", "pitch_type"])["is_barrel"].sum(),
    "hard_hits": bbe.groupby(["batter", "pitch_type"])["is_hard_hit"].sum(),
    "xwoba": bbe.groupby(["batter", "pitch_type"])["estimated_woba_using_speedangle"]
                .apply(lambda s: pd.to_numeric(s, errors="coerce").mean()),
})
damage["barrel_pct"] = damage["barrels"] / damage["bbe"]
damage["hard_hit_pct"] = damage["hard_hits"] / damage["bbe"]

lg_damage = pd.DataFrame({
    "lg_barrel": bbe.groupby("pitch_type")["is_barrel"].mean(),
    "lg_hard_hit": bbe.groupby("pitch_type")["is_hard_hit"].mean(),
    "lg_xwoba": bbe.groupby("pitch_type")["estimated_woba_using_speedangle"]
                   .apply(lambda s: pd.to_numeric(s, errors="coerce").mean()),
})
print(lg_damage.round(3).to_string())

            lg_barrel  lg_hard_hit  lg_xwoba
pitch_type                                  
CH              0.061         0.32     0.347
CS              0.062        0.188     0.297
CU              0.079         0.35     0.374
EP              0.072        0.383     0.366
FA              0.101        0.405     0.392
FC               0.08        0.374     0.375
FF                0.1        0.447     0.392
FO                0.0          0.3     0.400
FS              0.057        0.353     0.349
KC              0.065        0.369     0.372
KN              0.045        0.299     0.370
SC              0.091          0.5     0.415
SI              0.066        0.435     0.368
SL              0.072        0.348     0.361
ST              0.072        0.293     0.344
SV              0.075        0.325     0.363


In [2]:
JUDGE = 592450
MIN_BBE = 25

jd = damage.loc[JUDGE]
jd = jd[jd["bbe"] >= MIN_BBE].join(lg_damage)
jd["barrel_gap"] = jd["barrel_pct"] - jd["lg_barrel"]
jd["xwoba_gap"] = jd["xwoba"] - jd["lg_xwoba"]

print(jd[["bbe", "barrel_pct", "lg_barrel", "barrel_gap",
          "xwoba", "lg_xwoba", "xwoba_gap"]].round(3).to_string())

            bbe  barrel_pct  lg_barrel  barrel_gap     xwoba  lg_xwoba xwoba_gap
pitch_type                                                                      
CH           29       0.276      0.061       0.215  0.725004     0.347  0.378141
FC           35       0.229       0.08       0.149  0.516271     0.375  0.141265
FF          113       0.301        0.1       0.201  0.641966     0.392  0.249731
SI           87       0.276      0.066       0.209  0.641979     0.368  0.273826
SL           54       0.167      0.072       0.095  0.515351     0.361  0.153918
ST           42       0.333      0.072       0.261  0.663093     0.344  0.319181


In [3]:
# How often does "low whiff" coincide with "high damage"?
sw = f[f["is_swing"]]
whiff_split = pd.DataFrame({
    "swings": sw.groupby(["batter", "pitch_type"]).size(),
    "whiff_pct": sw.groupby(["batter", "pitch_type"])["is_whiff"].mean(),
})
lg_whiff = sw.groupby("pitch_type")["is_whiff"].mean()

combined = whiff_split.join(damage, how="inner")
combined = combined[(combined["swings"] >= 50) & (combined["bbe"] >= 25)]
combined = combined.join(lg_damage).join(lg_whiff.rename("lg_whiff"))

combined["whiff_gap"] = combined["whiff_pct"] - combined["lg_whiff"]
combined["barrel_gap"] = combined["barrel_pct"] - combined["lg_barrel"]

print(f"{len(combined)} (batter, pitch_type) pairs")
print("correlation whiff_gap vs barrel_gap:",
      round(combined["whiff_gap"].corr(combined["barrel_gap"]), 3))
print()

# The dangerous quadrant: hitter doesn't miss it AND punishes it
danger = combined[(combined["whiff_gap"] < 0) & (combined["barrel_gap"] > 0.05)]
print(f"low-whiff, high-damage pairs: {len(danger)} ({len(danger)/len(combined):.1%})")
print()
print(danger.nlargest(10, "barrel_gap")[
    ["swings", "whiff_gap", "bbe", "barrel_pct", "lg_barrel", "barrel_gap"]
].round(3).to_string())

1661 (batter, pitch_type) pairs
correlation whiff_gap vs barrel_gap: 0.318

low-whiff, high-damage pairs: 109 (6.6%)

                   swings  whiff_gap  bbe  barrel_pct  lg_barrel  barrel_gap
batter pitch_type                                                           
665742 FF             303     -0.014  125       0.296        0.1       0.196
671732 SL              89     -0.076   37       0.243      0.072       0.171
608369 SL             163     -0.010   56       0.232      0.072        0.16
665489 ST             117     -0.041   53       0.226      0.072       0.154
670541 SI             118     -0.032   74       0.216      0.066        0.15
660271 CH             136     -0.021   53       0.208      0.061       0.146
671277 ST              68     -0.003   28       0.214      0.072       0.142
663656 SI              68     -0.014   34       0.206      0.066       0.139
624585 FF             270     -0.041  106       0.236        0.1       0.136
671218 FC              77     -0.03

In [4]:
def approach_with_damage(batter_id, min_swings=50, min_bbe=25,
                         whiff_gap=0.05, barrel_gap=0.05):
    """Recommendations accounting for BOTH miss rate and damage.

    A pitch a hitter rarely misses but destroys is the most expensive
    mistake available, and a whiff-only report marks it as harmless.
    """
    recs = []

    try:
        w = whiff_split.loc[batter_id]
        w = w[w["swings"] >= min_swings].join(lg_whiff.rename("lg_whiff"))
        w["gap"] = w["whiff_pct"] - w["lg_whiff"]
    except KeyError:
        return ["INSUFFICIENT SAMPLE"]

    try:
        d = damage.loc[batter_id]
        d = d[d["bbe"] >= min_bbe].join(lg_damage)
        d["bgap"] = d["barrel_pct"] - d["lg_barrel"]
    except KeyError:
        d = pd.DataFrame()

    both = w.join(d[["bbe", "barrel_pct", "bgap"]], how="left")

    for pt, r in both.sort_values("gap", ascending=False).iterrows():
        if pd.notna(r.get("bgap")) and r["bgap"] > barrel_gap:
            if r["gap"] > whiff_gap:
                recs.append(
                    f"{pt}: chase pitch ONLY — whiffs {r['gap']:+.1%} but "
                    f"barrels {r['bgap']:+.1%} on contact "
                    f"({int(r['swings'])} sw / {int(r['bbe'])} bbe)")
            else:
                recs.append(
                    f"{pt}: AVOID — barrels {r['bgap']:+.1%} above league, "
                    f"whiffs only {r['gap']:+.1%} "
                    f"({int(r['swings'])} sw / {int(r['bbe'])} bbe)")
        elif r["gap"] > whiff_gap:
            recs.append(f"{pt}: attack — whiffs {r['gap']:+.1%} above league "
                        f"({int(r['swings'])} swings)")

    return recs or ["No significant deviations at these sample sizes"]

from src.data.player_ids import load_player_ids, display_name
ids = load_player_ids([592450, 665742, 660271])
names = display_name(ids)

for pid in [592450, 665742, 660271]:
    print(f"=== {names.get(pid, pid)} ===")
    for r in approach_with_damage(pid):
        print(f"  - {r}")
    print()

=== Judge, Aaron ===
  - CH: chase pitch ONLY — whiffs +16.5% but barrels +21.5% on contact (118 sw / 29 bbe)
  - CU: attack — whiffs +14.7% above league (61 swings)
  - SL: chase pitch ONLY — whiffs +9.9% but barrels +9.5% on contact (180 sw / 54 bbe)
  - FC: chase pitch ONLY — whiffs +7.9% but barrels +14.9% on contact (106 sw / 35 bbe)
  - ST: AVOID — barrels +26.1% above league, whiffs only +1.7% (121 sw / 42 bbe)
  - FF: AVOID — barrels +20.1% above league, whiffs only +1.1% (369 sw / 113 bbe)
  - SI: AVOID — barrels +20.9% above league, whiffs only +0.9% (199 sw / 87 bbe)

=== Soto, Juan ===
  - SI: AVOID — barrels +18.9% above league, whiffs only +1.0% (204 sw / 98 bbe)
  - FC: AVOID — barrels +10.9% above league, whiffs only -1.0% (118 sw / 53 bbe)
  - FF: AVOID — barrels +19.6% above league, whiffs only -1.4% (303 sw / 125 bbe)
  - CU: AVOID — barrels +7.5% above league, whiffs only -15.8% (51 sw / 26 bbe)

=== Ohtani, Shohei ===
  - FF: chase pitch ONLY — whiffs +5.0% but bar

In [5]:
print(damage.loc[(592450, "CU")] if (592450, "CU") in damage.index else "no CU damage row")

bbe                   17
barrels                4
hard_hits              9
xwoba           0.593968
barrel_pct      0.235294
hard_hit_pct    0.529412
Name: (592450, CU), dtype: object


In [6]:
def approach_final(batter_id, min_swings=50, min_bbe=25,
                   whiff_gap=0.05, barrel_gap=0.05):
    """As before, but distinguishes 'safe' from 'not measured'."""
    recs = []

    try:
        w = whiff_split.loc[batter_id]
        w = w[w["swings"] >= min_swings].join(lg_whiff.rename("lg_whiff"))
        w["gap"] = w["whiff_pct"] - w["lg_whiff"]
    except KeyError:
        return ["INSUFFICIENT SAMPLE"]

    try:
        d = damage.loc[batter_id].join(lg_damage)
        d["bgap"] = d["barrel_pct"] - d["lg_barrel"]
    except KeyError:
        d = pd.DataFrame(columns=["bbe", "bgap"])

    both = w.join(d[["bbe", "bgap"]], how="left")

    for pt, r in both.sort_values("gap", ascending=False).iterrows():
        bbe = r.get("bbe", np.nan)
        bgap = r.get("bgap", np.nan)
        measured = pd.notna(bbe) and bbe >= min_bbe

        if not measured:
            n = 0 if pd.isna(bbe) else int(bbe)
            if r["gap"] > whiff_gap:
                recs.append(f"{pt}: whiffs {r['gap']:+.1%} above league "
                            f"({int(r['swings'])} sw) — DAMAGE NOT MEASURED "
                            f"({n} bbe, need {min_bbe})")
            continue

        if bgap > barrel_gap:
            kind = "chase pitch ONLY" if r["gap"] > whiff_gap else "AVOID"
            recs.append(f"{pt}: {kind} — whiffs {r['gap']:+.1%}, "
                        f"barrels {bgap:+.1%} ({int(r['swings'])} sw / {int(bbe)} bbe)")
        elif r["gap"] > whiff_gap:
            recs.append(f"{pt}: ATTACK — whiffs {r['gap']:+.1%}, "
                        f"barrels {bgap:+.1%} ({int(r['swings'])} sw / {int(bbe)} bbe)")

    return recs or ["No significant deviations at these sample sizes"]

for pid in [592450, 665742, 660271]:
    print(f"=== {names.get(pid, pid)} ===")
    for r in approach_final(pid):
        print(f"  - {r}")
    print()

=== Judge, Aaron ===
  - CH: chase pitch ONLY — whiffs +16.5%, barrels +21.5% (118 sw / 29 bbe)
  - CU: whiffs +14.7% above league (61 sw) — DAMAGE NOT MEASURED (17 bbe, need 25)
  - SL: chase pitch ONLY — whiffs +9.9%, barrels +9.5% (180 sw / 54 bbe)
  - FC: chase pitch ONLY — whiffs +7.9%, barrels +14.9% (106 sw / 35 bbe)
  - ST: AVOID — whiffs +1.7%, barrels +26.1% (121 sw / 42 bbe)
  - FF: AVOID — whiffs +1.1%, barrels +20.1% (369 sw / 113 bbe)
  - SI: AVOID — whiffs +0.9%, barrels +20.9% (199 sw / 87 bbe)

=== Soto, Juan ===
  - SI: AVOID — whiffs +1.0%, barrels +18.9% (204 sw / 98 bbe)
  - FC: AVOID — whiffs -1.0%, barrels +10.9% (118 sw / 53 bbe)
  - FF: AVOID — whiffs -1.4%, barrels +19.6% (303 sw / 125 bbe)
  - CU: AVOID — whiffs -15.8%, barrels +7.5% (51 sw / 26 bbe)

=== Ohtani, Shohei ===
  - FF: chase pitch ONLY — whiffs +5.0%, barrels +16.2% (384 sw / 122 bbe)
  - SL: AVOID — whiffs +4.4%, barrels +23.8% (215 sw / 71 bbe)
  - ST: AVOID — whiffs +3.6%, barrels +16.8% (81 s